# Benchmark Knowledge Tracing — Analyse regroupée de tous les modeles

Ce notebook compare **sur les memes graphiques et dans les memes tables** tous
les modeles du projet : BKT, IRT statique, IRT dynamique,
PFA, GKT et GIRT.

## Principe : un artefact commun, les predictions out-of-fold (OOF)

Chaque modele utilise un decoupage, des donnees et des metriques differents. Pour
les comparer equitablement, on ne part **ni** des checkpoints `.pth` (il faudrait
encore les donnees + le code d'inference de chaque modele), **ni** des CSV de
metriques agregees (qui ne permettent pas de tracer une courbe), mais des
**predictions out-of-fold** de chaque modele, au schema unique :

| fold | y_true | y_pred | (skill) |
|------|--------|--------|---------|

A partir de ce seul fichier (`models/<modele>/oof_predictions.csv`), ce notebook
**recalcule toutes les metriques de maniere identique** pour tous les modeles, et
trace les courbes superposees.

> **Generer les predictions** (a faire une fois, avec les donnees) :
> ```bash
> python -m analysis.exporters.run_all      # tous les modeles disponibles
> # ou un par un, p.ex. :
> python -m analysis.exporters.export_dynamic_irt
> ```

## Avertissement methodologique important

Les modeles ne sont pas tous evalues dans le **meme regime** :

- **Cold-start, student-wise CV** (eleves de test jamais vus) : BKT, IRT
  dynamique, GIRT, GKT, PFA. C'est le scenario realiste d'onboarding d'un nouvel
  eleve, et le plus exigeant.
- **Warm-start, splits repetes** (l'eleve / la question a ete vu a
  l'entrainement) : IRT statique (py-irt apprend une capacite par eleve).


## 0. Configuration

In [ ]:
import sys, os
from pathlib import Path

# Marche que le notebook soit lance depuis la racine du projet ou depuis analysis/
CWD = Path.cwd()
ROOT = CWD if (CWD / "models").is_dir() else CWD.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "analysis"))
os.chdir(ROOT)            # pour que les chemins "models/..." soient corrects

import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import benchmark_utils as bu
importlib.reload(bu)

FIG_DIR = ROOT / "analysis" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
})
pd.set_option("display.width", 160)
print("Racine projet :", ROOT)

## 1. Chargement des predictions

On charge tous les `oof_predictions.csv` disponibles. **Si aucun n'est present**
(par exemple ici sans les donnees), le notebook bascule sur un **jeu synthetique
de demonstration** pour que toutes les figures s'affichent — un bandeau le
signale alors clairement.

In [ ]:
oof_dict = bu.load_all_oof()
DEMO = len(oof_dict) == 0

if DEMO:
    print("\n" + "!"*72)
    print("!  MODE DEMO : aucun oof_predictions.csv trouve.")
    print("!  Donnees SYNTHETIQUES affichees pour montrer la structure du notebook.")
    print("!  Lancez 'python -m analysis.exporters.run_all' avec vos donnees,")
    print("!  puis re-executez ce notebook pour les vrais resultats.")
    print("!"*72)

    rng = np.random.default_rng(42)
    # forces relatives plausibles (juste pour la demo, sans valeur scientifique)
    strengths = {"bkt": 0.85, "irt": 1.0, "dynamic_irt": 1.35,
                 "pfa": 1.15, "optimized_pfa": 1.2, "gkt": 1.1, "girt": 1.25}
    base_rate = 0.7
    oof_dict = {}
    for key, s in strengths.items():
        parts = []
        for f in range(5):
            n = 4000
            y = rng.binomial(1, base_rate, n)
            logit = s * (y - 0.5) * 2.4 + rng.normal(0, 1.0, n) + 0.15 * (f - 2)
            p = 1 / (1 + np.exp(-logit))
            parts.append(pd.DataFrame({"fold": f, "y_true": y, "y_pred": p,
                                       "skill": rng.integers(0, 12, n)}))
        oof_dict[key] = pd.concat(parts, ignore_index=True)

print(f"\nModeles charges : {[bu.REGISTRY_BY_KEY[k].display_name for k in oof_dict]}")

## 2. Disponibilite et volume par modele

In [ ]:
rows = []
for spec in bu.MODEL_REGISTRY:
    if spec.key in oof_dict:
        oof = oof_dict[spec.key]
        rows.append({"Modele": spec.display_name, "Famille": spec.family,
                     "Statut": "OK" if not DEMO else "DEMO",
                     "Folds": oof["fold"].nunique(), "Observations": len(oof),
                     "Taux succes": round(oof["y_true"].mean(), 3)})
    else:
        rows.append({"Modele": spec.display_name, "Famille": spec.family,
                     "Statut": "absent", "Folds": 0, "Observations": 0,
                     "Taux succes": np.nan})
avail = pd.DataFrame(rows).set_index("Modele")
avail

## 3. Table comparative des metriques

Toutes les metriques sont **recalculees identiquement** depuis les predictions :

- **AUC** : discrimination (plus haut = mieux).
- **RMSE**, **LogLoss** : erreur probabiliste (plus bas = mieux).

D'abord la **moyenne ± ecart-type sur les folds**, puis la version **pooled**
(metriques sur l'ensemble des predictions concatenees).

In [ ]:
table_meanstd = bu.build_comparison_table(oof_dict, kind="mean_std")
table_meanstd[bu.METRIC_ORDER]

In [ ]:
table_pooled = bu.build_comparison_table(oof_dict, kind="pooled")
table_pooled[bu.METRIC_ORDER]

### Intervalles de confiance bootstrap (AUC)

Chaque modele ayant un jeu de test different, on ne fait pas de test apparie
inter-modeles : on regarde si les **IC a 95 %** (bootstrap sur les predictions
de chaque modele) se **chevauchent**.

In [ ]:
ci_rows = []
for key, oof in oof_dict.items():
    pt, lo, hi = bu.bootstrap_metric_ci(oof["y_true"].values, oof["y_pred"].values,
                                        metric="AUC", n_boot=500)
    ci_rows.append({"Modele": bu.REGISTRY_BY_KEY[key].display_name,
                    "AUC": round(pt, 4), "IC bas": round(lo, 4), "IC haut": round(hi, 4)})
pd.DataFrame(ci_rows).sort_values("AUC", ascending=False).set_index("Modele")

## 4. Courbes ROC regroupees

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6.5))
bu.plot_roc_all(oof_dict, ax=ax)
fig.savefig(FIG_DIR / "roc_all_models.pdf", bbox_inches="tight")
plt.show()

## 5. Comparaison metrique par metrique

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
for ax, metric in zip(axes.ravel(), ["AUC", "RMSE", "LogLoss"]):
    bu.plot_metric_bars(oof_dict, metric=metric, ax=ax)
fig.tight_layout()
fig.savefig(FIG_DIR / "metric_bars.pdf", bbox_inches="tight")
plt.show()

## 6. Stabilite inter-fold

Un bon modele n'est pas seulement performant en moyenne, il est aussi **stable**
d'un fold a l'autre. Une forte dispersion signale une sensibilite au decoupage
(et donc une estimation moins fiable).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
bu.plot_fold_dispersion(oof_dict, metric="AUC", ax=axes[0])
bu.plot_fold_dispersion(oof_dict, metric="LogLoss", ax=axes[1])
fig.tight_layout()
fig.savefig(FIG_DIR / "fold_dispersion.pdf", bbox_inches="tight")
plt.show()

## 7. Regularite des performances par competence

Pour chaque modele disposant d'une colonne `skill`, on calcule l'AUC **par
competence** puis on regarde la **distribution** de ces AUC. Une boite resserree
et haute = le modele predit bien *partout* ; une boite etalee = des competences
mal modelisees.

> Les identifiants de competence ne sont pas alignables entre modeles (encodages
> differents) : on compare donc des *distributions intra-modele*, pas competence
> par competence.

In [ ]:
per_skill = {}
for key, oof in oof_dict.items():
    ps = bu.per_skill_metric(oof, metric="AUC", min_obs=30)
    if ps is not None and len(ps) >= 3:
        per_skill[key] = ps["AUC"].values

if per_skill:
    fig, ax = plt.subplots(figsize=(10, 5.5))
    labels = [bu.REGISTRY_BY_KEY[k].display_name for k in per_skill]
    ax.boxplot(list(per_skill.values()), labels=labels, showmeans=True)
    ax.axhline(0.5, ls="--", color="grey", lw=1, label="Aleatoire")
    ax.set_ylabel("AUC par competence")
    ax.set_title("Distribution de l'AUC par competence (intra-modele)")
    ax.tick_params(axis="x", rotation=30)
    for lbl in ax.get_xticklabels():
        lbl.set_ha("right")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / "per_skill_auc.pdf", bbox_inches="tight")
    plt.show()
else:
    print("Aucun modele avec colonne 'skill' exploitable.")

## 9. Export des tables (pour le rapport)

In [ ]:
out_csv = ROOT / "analysis" / "tables"
out_csv.mkdir(exist_ok=True)
bu.build_comparison_table(oof_dict, "mean_std")[bu.METRIC_ORDER].to_csv(out_csv / "metrics_mean_std.csv")
bu.build_comparison_table(oof_dict, "pooled")[bu.METRIC_ORDER].to_csv(out_csv / "metrics_pooled.csv")
print("Tables ecrites dans", out_csv)
print("Figures ecrites dans", FIG_DIR)
if DEMO:
    print("\n(MODE DEMO — chiffres synthetiques, sans valeur scientifique.)")

---
### Recapitulatif des limites

1. **Heterogeneite des regimes** (warm vs cold start) : ne pas lire le classement
   au pied de la lettre entre les deux familles.
2. **Decoupages differents** entre modeles (student-wise simple, balanced
   student-wise, KFold sur users, splits par competence/question). Les `fold` ne
   recouvrent pas exactement les memes eleves d'un modele a l'autre.
3. **GKT** : seul le fold 0 est entraine pour l'instant — entrainer les folds 1-4
   pour une moyenne sur 5 folds comparable aux autres.
